<a href="https://colab.research.google.com/github/PravinV001/Python/blob/Class_N_Colab/Sklearn_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Below code is not good and can't be taken in prod becz of data leakage and error prone while moving into production.

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

# Step 1: Load and split
X_train, X_test, y_train, y_test = train_test_split(X, y)

# Step 2: Scale numerical features... manually
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[num_cols])
X_test_scaled = scaler.transform(X_test[num_cols])  # EASY TO FORGET!

# Step 3: Encode categorical features... manually
encoder = OneHotEncoder()
X_train_encoded = encoder.fit_transform(X_train[cat_cols])
X_test_encoded = encoder.transform(X_test[cat_cols])

# Step 4: Stitch everything back together... manually
X_train_final = np.hstack([X_train_scaled, X_train_encoded])
X_test_final = np.hstack([X_test_scaled, X_test_encoded])

# Step 5: Train
model = LinearRegression()
model.fit(X_train_final, y_train)
model.predict(X_test_final)


In [ ]:
# !pip install -U gdown

In [ ]:
!gdown  1Zim4ODFec7nQqx5W_l8XzwzC_IB-3rmq

Downloading...
From: https://drive.google.com/uc?id=1Zim4ODFec7nQqx5W_l8XzwzC_IB-3rmq
To: /content/CarPrice_Assignment.csv
100% 26.7k/26.7k [00:00<00:00, 40.3MB/s]


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer          # NEW!
from sklearn.pipeline import Pipeline                   # NEW!

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings("ignore")


In [ ]:
# Load the dataset
pd.set_option('display.max_columns',None)
cars = pd.read_csv("CarPrice_Assignment.csv")
print("Shape:", cars.shape)
cars.head()


Shape: (205, 26)


,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,carheight,curbweight,enginetype,cylindernumber,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,171.2,65.5,52.4,2823,ohcv,six,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,176.6,66.2,54.3,2337,ohc,four,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,176.6,66.4,54.3,2824,ohc,five,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


In [ ]:
cars['symboling'].value_counts()

,count
symboling,
0,67
1,54
2,32
3,27
-1,22
-2,3


In [ ]:
cars['CarName'].value_counts()


,count
CarName,
peugeot 504,6
toyota corolla,6
toyota corona,6
subaru dl,4
mitsubishi outlander,3
...,...
volkswagen super beetle,1
volkswagen rabbit custom,1
volvo 245,1


In [ ]:
cars["car_company"] = (cars["CarName"]
                              .str.split(" ").str[0].str.lower())

In [ ]:
cars["car_company"].value_counts()

,count
car_company,
toyota,31
nissan,18
mazda,15
honda,13
mitsubishi,13
subaru,12
peugeot,11
volvo,11
volkswagen,9


In [ ]:
cars_clean = cars.copy()

In [ ]:
# Treat symboling as categorical (it's a discrete risk index, not continuous)
cars_clean["symboling"] = cars_clean["symboling"].astype("object")

# Extract car company from CarName
cars_clean["car_company"] = (cars_clean["CarName"]
                              .str.split(" ").str[0].str.lower())

# Fix common typos in company names
make_fixes = {
    "vw": "volkswagen", "vokswagen": "volkswagen",
    "porcshce": "porsche", "toyouta": "toyota", "maxda": "mazda",
}
cars_clean["car_company"] = cars_clean["car_company"].replace(make_fixes)

# Drop raw text and ID columns
cars_clean = cars_clean.drop(columns=["CarName", "car_ID"])


In [ ]:
# Features / target
X = cars_clean.drop(columns=["price"])
y = cars_clean["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (143, 24) Test: (62, 24)


In [ ]:
num_cols = X_train.select_dtypes(include=["int64","float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))
print("\nExample numeric:", num_cols[:5])
print("Example categorical:", cat_cols[:5])

Numeric columns: 13
Categorical columns: 11

Example numeric: ['wheelbase', 'carlength', 'carwidth', 'carheight', 'curbweight']
Example categorical: ['symboling', 'fueltype', 'aspiration', 'doornumber', 'carbody']


In [ ]:
from sklearn.pipeline import Pipeline

simple_pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),        # Step 1: Transformer
    ("model", LinearRegression())        # Step 2: Estimator (last step)
])

# Use ONLY numeric columns for now
simple_pipe.fit(X_train[num_cols], y_train)
pred_test = simple_pipe.predict(X_test[num_cols])

print("Test R2:", r2_score(y_test, pred_test))

Test R2: 0.797496570420891


In [ ]:
toy = pd.DataFrame({
    "age": [18, 25, 40, 35],
    "income": [20_000, 45_000, 80_000, 60_000],
    "city": ["delhi", "delhi", "mumbai", "jaipur"],
    "owns_car": ["no", "yes", "yes", "no"],
})

toy_num = ["age", "income"]
toy_cat = ["city", "owns_car"]

print(toy)


   age  income    city owns_car
0   18   20000   delhi       no
1   25   45000   delhi      yes
2   40   80000  mumbai      yes
3   35   60000  jaipur       no


In [ ]:
# Manual Step 1: Scale numeric columns
scaler = StandardScaler()
toy_num_scaled = scaler.fit_transform(toy[toy_num])
# 1 more step will be added when we will have test data as well
toy_num_scaled_df = pd.DataFrame(toy_num_scaled,
                                  columns=[f"scaled_{c}" for c in toy_num])

# Manual Step 2: One-hot encode categorical columns
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
toy_cat_encoded = encoder.fit_transform(toy[toy_cat])
# 1 more step will be added when we will have test data as well

toy_cat_encoded_df = pd.DataFrame(toy_cat_encoded,
                                   columns=encoder.get_feature_names_out(toy_cat))

# Manual Step 3: Stitch back together
toy_manual = pd.concat([toy_num_scaled_df, toy_cat_encoded_df], axis=1)
print(toy_manual)


   scaled_age  scaled_income  city_delhi  city_jaipur  city_mumbai  \
0   -1.343674      -1.426825         1.0          0.0          0.0   
1   -0.525786      -0.285365         1.0          0.0          0.0   
2    1.226833       1.312679         0.0          0.0          1.0   
3    0.642627       0.399511         0.0          1.0          0.0   

   owns_car_no  owns_car_yes  
0          1.0           0.0  
1          0.0           1.0  
2          0.0           1.0  
3          1.0           0.0  


In [ ]:
from sklearn.compose import ColumnTransformer

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

toy_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), toy_num),    # Scale these columns
        ("cat", ohe, toy_cat)                   # Encode these columns
    ],
    remainder="drop"                            # Drop any other columns
).set_output(transform="pandas")                # Get nice DataFrame output

toy_transformed = toy_preprocess.fit_transform(toy)
# transform - test data
print(toy_transformed)


   num__age  num__income  cat__city_delhi  cat__city_jaipur  cat__city_mumbai  \
0 -1.343674    -1.426825              1.0               0.0               0.0   
1 -0.525786    -0.285365              1.0               0.0               0.0   
2  1.226833     1.312679              0.0               0.0               1.0   
3  0.642627     0.399511              0.0               1.0               0.0   

   cat__owns_car_no  cat__owns_car_yes  
0               1.0                0.0  
1               0.0                1.0  
2               0.0                1.0  
3               1.0                0.0  


In [ ]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", ohe, cat_cols),
    ],
    remainder="drop",
).set_output(transform="pandas")

# Demonstrate: fit on train, transform test
X_train_pp = preprocess.fit_transform(X_train)
X_test_pp = preprocess.transform(X_test)

print("Preprocessed train shape:", X_train_pp.shape)
print("Preprocessed test shape :", X_test_pp.shape)


Preprocessed train shape: (143, 77)
Preprocessed test shape : (62, 77)


In [ ]:
linreg_pipe = Pipeline(steps=[("preprocess", preprocess),        # ColumnTransformer handles ALL preprocessing
    ("model", LinearRegression())      # Final estimator
])

linreg_pipe.fit(X_train, y_train) #transformation - learn weights

pred_train = linreg_pipe.predict(X_train) # predn on train
pred_test  = linreg_pipe.predict(X_test) # predn on test

print("Linear Regression")
print("  Train R2:", r2_score(y_train, pred_train))
print("  Test  R2:", r2_score(y_test, pred_test))
print("  Test MSE:", mean_squared_error(y_test, pred_test))


Linear Regression
  Train R2: 0.9786249443259047
  Test  R2: 0.8890733062476718
  Test MSE: 7685487.048033528


In [ ]:
def eval_regression(pipe, name: str):
    """Train a pipeline and return evaluation metrics."""
    pipe.fit(X_train, y_train)
    pred_test = pipe.predict(X_test)
    return {
        "model": name,
        "train_r2": r2_score(y_train, pipe.predict(X_train)),
        "test_r2": r2_score(y_test, pred_test),
        "test_mse": mean_squared_error(y_test, pred_test),
    }


In [ ]:
# Build multiple pipelines with DIFFERENT models, SAME preprocessing
linreg_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LinearRegression())
])

ridge_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", Ridge(alpha=10.0, random_state=42))
])

# Compare them
results = [
    eval_regression(linreg_pipe, "LinearRegression"),
    eval_regression(ridge_pipe, "Ridge(alpha=10)"),
]

pd.DataFrame(results)


,model,train_r2,test_r2,test_mse
0,LinearRegression,0.978625,0.889073,7.685487e+06
1,Ridge(alpha=10),0.935029,0.851998,1.025424e+07


In [ ]:
# K-Fold setup
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Create a pipeline for Ridge tuning
ridge_tune_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", Ridge())
])

# Define hyperparameter grid
# IMPORTANT: Use "model__alpha" (double underscore!)
# This tells GridSearchCV: "access the 'model' step, set its 'alpha' parameter"
ridge_param_grid = {
    "model__alpha": [0.001, 0.01, 0.1, 1, 3, 10, 30, 100, 300, 1000]
}

# Run Grid Search with Cross-Validation
ridge_grid = GridSearchCV(
    estimator=ridge_tune_pipe,
    param_grid=ridge_param_grid,
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)

ridge_grid.fit(X_train, y_train)

print("Best Ridge alpha:", ridge_grid.best_params_["model__alpha"])
print("Best CV score:", ridge_grid.best_score_)


Best Ridge alpha: 1
Best CV score: -6311415.875947319


In [ ]:
# The best model is already trained!
best_ridge = ridge_grid.best_estimator_
pred_test = best_ridge.predict(X_test)

print("Best Ridge")
print("  Train R2:", r2_score(y_train, best_ridge.predict(X_train)))
print("  Test  R2:", r2_score(y_test, pred_test))
print("  Test MSE:", mean_squared_error(y_test, pred_test))


Best Ridge
  Train R2: 0.9709844998243002
  Test  R2: 0.8912227413356884
  Test MSE: 7536564.773595025
